# Data check

Load a configured dataset and verify the series-to-event conversion semantics.

In [1]:
from pathlib import Path
import sys

# Use local package when running from a fresh checkout.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))

In [2]:
from afc_robustness.experiment import BenchmarkConfig, load_dataset_from_config
from afc_robustness.data import describe_dataset
from afc_robustness.representations import series_to_episode

config_path = repo_root / "configs" / "fcc.yaml"  # smoke, fcc, tep
cfg = BenchmarkConfig.from_yaml(config_path)

Generate or place data before running this cell. For a smoke dataset, run `python scripts/create_synthetic_dataset.py --output data/smoke` from the repository root.

In [3]:
dataset = load_dataset_from_config(cfg)
print(dataset)
describe_dataset(dataset)

AlarmSeriesDataset(X=array([[[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 1, 1, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 1, 1, 1],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 1, 1, 1],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 1, 1, 1],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       ...,

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
  

,class_label,class_name,n_episodes,min_length,max_length,mean_length
0,0,catalyst_deactivation,100,60,60,60.0
1,1,cyclone_damage,100,60,60,60.0
2,2,preheater_shutdown,100,60,60,60.0
3,3,preheater_temp_increase,100,60,60,60.0
4,4,V2_high,100,60,60,60.0
5,5,V2_low,100,60,60,60.0
6,6,V3_high,100,60,60,60.0
7,7,V3_low,100,60,60,60.0
8,8,V4_high,100,60,60,60.0
9,9,V4_low,100,60,60,60.0


In [4]:
idx = 0
episode = dataset.episode(idx)
print("sample_id:", episode.sample_id)
print("n_events:", episode.n_events)
print("horizon:", episode.horizon)
print("first events:", episode.events[:10])

sample_id: catalyst_deactivation/catalyst_deactivation_run100_alarm
n_events: 14
horizon: 59.0
first events: (AlarmEvent(tag=30, event_type=<EventType.ACT: 'ACT'>, timestamp=0.0, order=6), AlarmEvent(tag=52, event_type=<EventType.ACT: 'ACT'>, timestamp=0.0, order=12), AlarmEvent(tag=2, event_type=<EventType.ACT: 'ACT'>, timestamp=1.0, order=0), AlarmEvent(tag=45, event_type=<EventType.ACT: 'ACT'>, timestamp=6.0, order=9), AlarmEvent(tag=25, event_type=<EventType.ACT: 'ACT'>, timestamp=7.0, order=4), AlarmEvent(tag=48, event_type=<EventType.ACT: 'ACT'>, timestamp=7.0, order=10), AlarmEvent(tag=26, event_type=<EventType.ACT: 'ACT'>, timestamp=10.0, order=5), AlarmEvent(tag=21, event_type=<EventType.ACT: 'ACT'>, timestamp=13.0, order=2), AlarmEvent(tag=22, event_type=<EventType.ACT: 'ACT'>, timestamp=16.0, order=3), AlarmEvent(tag=43, event_type=<EventType.ACT: 'ACT'>, timestamp=23.0, order=8))


In [5]:
n_episodes = dataset.n_episodes
act_counts = []

for i in range(n_episodes):
    ep = dataset.episode(i)
    n_act = sum(getattr(event.event_type, "value", event.event_type) == "ACT" for event in ep.events)
    act_counts.append(n_act)

total_act_events = sum(act_counts)
min_act = min(act_counts)
max_act = max(act_counts)
mean_act = total_act_events / n_episodes

print(f"Number of episodes: {n_episodes}")
print(f"Total alarm activation events: {total_act_events}")
print(f"Activation events per episode range: {min_act} to {max_act}")
print(f"Mean activation events per episode: {mean_act:.2f}")

Number of episodes: 1600
Total alarm activation events: 20789
Activation events per episode range: 2 to 69
Mean activation events per episode: 12.99
